# MOSS-Image / ComfyUI 一键启动

选择 GPU 运行时，然后点击「全部运行」。首次需要在 Google 提示中完成 Drive 授权。

- MOSS 与六款 ANIMA 共约 39.55 GB 权重，校验后存到 `我的云端硬盘/ComfyUI_Colab/models`。首次需要下载；官方组件共用一份。
- 准备单元可选“全部”（默认）、“仅 ANIMA”或“仅 MOSS”；日常出图在客户端选择具体模型。
- Google 免费云盘通常只有 15 GB；整套权重约 40 GB，需要足够的永久存储额度。
- 每次新运行时先从 Drive 恢复代码和模型到本地磁盘；同一运行时重复执行会复用已校验文件与服务。
- ComfyUI 固定到 2026-10-26 检查的最新提交；原目录、大模型缓存和图片保留。
- 图片、上传素材和工作流分别保存在 `ComfyUI_Colab/output`、`input` 和 `user`。
- 最后一格会创建临时 HTTPS 网页与 API 入口，并显示访问密钥。API 使用 `Authorization: Bearer <密钥>`；网页使用密钥登录。
- 地址和密钥仅在本次运行时有效。不要分享密钥或连接配置。笔记本不保存单元格输出，重开后可重跑最后一格查看当前连接信息。
- 云端接口说明会保存到 `ComfyUI_Colab/api/CLOUD_API.md`。本地提示词工具与配套工作流由另一台电脑单独适配。

首次启动包含下载与安装，需要等待。Colab 断开并回收运行时后，服务会停止；Drive 中的文件保留。


In [ ]:
# 1. 挂载 Drive。Google 首次授权提示仍需要您确认。
MOSS_READY = False
from google.colab import drive
from pathlib import Path
try:
    drive.mount('/content/drive')
except Exception as error:
    raise RuntimeError('Drive 挂载失败。请在 Colab 完成 Google 授权；如持续失败，请重新连接运行时后重试。此时尚未下载模型，也未启动 ComfyUI。') from error
if not Path('/content/drive/MyDrive').is_dir():
    raise RuntimeError('Drive 挂载未就绪，请先完成 Google 授权。')


In [ ]:
#@title 2. 准备模型库
MODEL_GROUP = "全部" #@param ["全部", "仅 ANIMA", "仅 MOSS"]
import json
from pathlib import PurePosixPath

MODEL_GROUPS={'全部':None,'仅 ANIMA':'ANIMA','仅 MOSS':'MOSS'}

def select_model_assets(registry, group):
    if group not in MODEL_GROUPS:
        raise ValueError('未知模型组，请选择全部、仅 ANIMA 或仅 MOSS。')
    if registry.get('schema_version')!=1:
        raise ValueError('模型目录版本不受支持。')
    files=registry['files']; models=registry['models']
    asset_map={item['path']:item for item in files}
    if len(asset_map)!=len(files) or len({m['id'] for m in models})!=len(models):
        raise ValueError('模型目录含重复的路径或 ID。')
    for path in asset_map:
        p=PurePosixPath(path)
        if p.is_absolute() or '..' in p.parts or '\\' in path:
            raise ValueError('模型目录含非法资产路径。')
    family=MODEL_GROUPS[group]
    selected=[m for m in models if family is None or m['family']==family]
    if not selected: raise ValueError('本次模型组为空。')
    required={path for m in selected for path in m['assets']}
    if not required.issubset(asset_map): raise ValueError('模型目录缺少依赖资产。')
    return [m['id'] for m in selected],[f for f in files if f['path'] in required]

MODEL_REGISTRY = json.loads('{\n  "schema_version": 1,\n  "checked_at": "2026-10-26",\n  "models": [\n    {\n      "id": "moss_image_sft_int8",\n      "display_name": "MOSS-Image INT8",\n      "family": "MOSS",\n      "version": "SFT INT8",\n      "checkpoint": "diffusion_models/moss_image_sft_int8_convrot.safetensors",\n      "text_encoder": "text_encoders/qwen3.5_9b_w4a8_gptq.safetensors",\n      "vae": "vae/flux2-vae_small.safetensors",\n      "assets": [\n        "diffusion_models/moss_image_sft_int8_convrot.safetensors",\n        "text_encoders/qwen3.5_9b_w4a8_gptq.safetensors",\n        "vae/flux2-vae_small.safetensors"\n      ],\n      "suggested_defaults": {\n        "steps": 25,\n        "cfg": 2.5,\n        "sampler": "er_sde",\n        "scheduler": "simple"\n      },\n      "notes": ""\n    },\n    {\n      "id": "anima_turbo_v1_1",\n      "display_name": "ANIMA Turbo v1.1",\n      "family": "ANIMA",\n      "version": "v1.1",\n      "checkpoint": "diffusion_models/anima-turbo-v1.1.safetensors",\n      "text_encoder": "text_encoders/qwen_3_06b_base.safetensors",\n      "vae": "vae/qwen_image_vae.safetensors",\n      "assets": [\n        "diffusion_models/anima-turbo-v1.1.safetensors",\n        "text_encoders/qwen_3_06b_base.safetensors",\n        "vae/qwen_image_vae.safetensors"\n      ],\n      "suggested_defaults": {\n        "steps": 10,\n        "cfg": 1,\n        "sampler": "euler",\n        "scheduler": "normal"\n      },\n      "notes": "快速预览；蒸馏版。"\n    },\n    {\n      "id": "anima_aesthetic_v1_1",\n      "display_name": "ANIMA Aesthetic v1.1",\n      "family": "ANIMA",\n      "version": "v1.1",\n      "checkpoint": "diffusion_models/anima-aesthetic-v1.1.safetensors",\n      "text_encoder": "text_encoders/qwen_3_06b_base.safetensors",\n      "vae": "vae/qwen_image_vae.safetensors",\n      "assets": [\n        "diffusion_models/anima-aesthetic-v1.1.safetensors",\n        "text_encoders/qwen_3_06b_base.safetensors",\n        "vae/qwen_image_vae.safetensors"\n      ],\n      "suggested_defaults": {\n        "steps": 35,\n        "cfg": 4.5,\n        "sampler": "er_sde",\n        "scheduler": "normal"\n      },\n      "notes": "默认插画审美；官方版本。"\n    },\n    {\n      "id": "anima_base_v1",\n      "display_name": "ANIMA Base v1.0",\n      "family": "ANIMA",\n      "version": "v1.0",\n      "checkpoint": "diffusion_models/anima-base-v1.0.safetensors",\n      "text_encoder": "text_encoders/qwen_3_06b_base.safetensors",\n      "vae": "vae/qwen_image_vae.safetensors",\n      "assets": [\n        "diffusion_models/anima-base-v1.0.safetensors",\n        "text_encoders/qwen_3_06b_base.safetensors",\n        "vae/qwen_image_vae.safetensors"\n      ],\n      "suggested_defaults": {\n        "steps": 35,\n        "cfg": 4.5,\n        "sampler": "er_sde",\n        "scheduler": "normal"\n      },\n      "notes": "官方基础模型；用于风格、LoRA 对照。"\n    },\n    {\n      "id": "wai_anima_v1_0",\n      "display_name": "WAI-ANIMA v1.0",\n      "family": "ANIMA",\n      "version": "v1.0 Base 1.0",\n      "checkpoint": "diffusion_models/waiANIMA_v10Base10.safetensors",\n      "text_encoder": "text_encoders/qwen_3_06b_base.safetensors",\n      "vae": "vae/qwen_image_vae.safetensors",\n      "assets": [\n        "diffusion_models/waiANIMA_v10Base10.safetensors",\n        "text_encoders/qwen_3_06b_base.safetensors",\n        "vae/qwen_image_vae.safetensors"\n      ],\n      "suggested_defaults": {\n        "steps": 25,\n        "cfg": 4.5,\n        "sampler": "euler_ancestral",\n        "scheduler": "normal"\n      },\n      "notes": "社区模型；镜像字节大小与完整 SHA-256 对照作者文件。"\n    },\n    {\n      "id": "animayume_v1_5_base",\n      "display_name": "AnimaYume v1.5 Base",\n      "family": "ANIMA",\n      "version": "v1.5 Base",\n      "checkpoint": "diffusion_models/AnimaYume_v15_base.safetensors",\n      "text_encoder": "text_encoders/qwen_3_06b_base.safetensors",\n      "vae": "vae/qwen_image_vae.safetensors",\n      "assets": [\n        "diffusion_models/AnimaYume_v15_base.safetensors",\n        "text_encoders/qwen_3_06b_base.safetensors",\n        "vae/qwen_image_vae.safetensors"\n      ],\n      "suggested_defaults": {\n        "steps": 30,\n        "cfg": 5,\n        "sampler": "euler_ancestral",\n        "scheduler": "normal"\n      },\n      "notes": "复杂描述；旧版 LoRA 需要重新验证。"\n    },\n    {\n      "id": "miaomiao_harem_anima_v1_6",\n      "display_name": "MiaoMiao Harem ANIMA v1.6",\n      "family": "ANIMA",\n      "version": "Anima 1.6",\n      "checkpoint": "diffusion_models/miaomiaoHarem_anima16.safetensors",\n      "text_encoder": "text_encoders/qwen_3_06b_base.safetensors",\n      "vae": "vae/qwen_image_vae.safetensors",\n      "assets": [\n        "diffusion_models/miaomiaoHarem_anima16.safetensors",\n        "text_encoders/qwen_3_06b_base.safetensors",\n        "vae/qwen_image_vae.safetensors"\n      ],\n      "suggested_defaults": {\n        "steps": 35,\n        "cfg": 4.5,\n        "sampler": "er_sde",\n        "scheduler": "normal"\n      },\n      "notes": "保留已采用的 2B 分支，不自动替换为 2.9B Beta。"\n    }\n  ],\n  "files": [\n    {\n      "path": "diffusion_models/moss_image_sft_int8_convrot.safetensors",\n      "repo_id": "Comfy-Org/MOSS-Image",\n      "repo_file": "diffusion_models/moss_image_sft_int8_convrot.safetensors",\n      "revision": "24147db865fcdc207270f844777587223fed8d6e",\n      "size_bytes": 5413698958,\n      "sha256": "b4d1475bc7eeffd7b809e82ea13a1fba1fb85c4e18365af3813722faa5c0fe4f"\n    },\n    {\n      "path": "text_encoders/qwen3.5_9b_w4a8_gptq.safetensors",\n      "repo_id": "Comfy-Org/MOSS-Image",\n      "repo_file": "text_encoders/qwen3.5_9b_w4a8_gptq.safetensors",\n      "revision": "24147db865fcdc207270f844777587223fed8d6e",\n      "size_bytes": 7343591088,\n      "sha256": "4cc6a1543ad2061806da46f35f90cadb4801dcba246c44782cf42ec0feb6407c"\n    },\n    {\n      "path": "vae/flux2-vae_small.safetensors",\n      "repo_id": "Comfy-Org/MOSS-Image",\n      "repo_file": "vae/flux2-vae_small.safetensors",\n      "revision": "24147db865fcdc207270f844777587223fed8d6e",\n      "size_bytes": 249521340,\n      "sha256": "d8d52ba036475f5fb07c8b435e176d3d97ebfa82f0d1a1c317f9cc1e25bd013b"\n    },\n    {\n      "path": "diffusion_models/anima-base-v1.0.safetensors",\n      "repo_id": "circlestone-labs/Anima",\n      "repo_file": "split_files/diffusion_models/anima-base-v1.0.safetensors",\n      "revision": "f973fc41ec7545364ac9776c2440285f43ff2a30",\n      "size_bytes": 4182218328,\n      "sha256": "bd43b7cffe1ed1153d9c41e7beb2f18cb1273eafbaa3af3edd6a173dc90a006e"\n    },\n    {\n      "path": "diffusion_models/anima-aesthetic-v1.1.safetensors",\n      "repo_id": "circlestone-labs/Anima",\n      "repo_file": "split_files/diffusion_models/anima-aesthetic-v1.1.safetensors",\n      "revision": "f973fc41ec7545364ac9776c2440285f43ff2a30",\n      "size_bytes": 4182230656,\n      "sha256": "3c1868387a3a1ff504bbb87c33678321965ead381fcf87afbd0264daa600c082"\n    },\n    {\n      "path": "diffusion_models/anima-turbo-v1.1.safetensors",\n      "repo_id": "circlestone-labs/Anima",\n      "repo_file": "split_files/diffusion_models/anima-turbo-v1.1.safetensors",\n      "revision": "f973fc41ec7545364ac9776c2440285f43ff2a30",\n      "size_bytes": 4182230656,\n      "sha256": "fba11953276b57edf59d1dc4f1857ac05aa079c56f982b4d7c20298d57d3f7eb"\n    },\n    {\n      "path": "text_encoders/qwen_3_06b_base.safetensors",\n      "repo_id": "circlestone-labs/Anima",\n      "repo_file": "split_files/text_encoders/qwen_3_06b_base.safetensors",\n      "revision": "f973fc41ec7545364ac9776c2440285f43ff2a30",\n      "size_bytes": 1192135096,\n      "sha256": "cd2a512003e2f9f3cd3c32a9c3573f820bb28c940f73c57b1ddaa983d9223eba"\n    },\n    {\n      "path": "vae/qwen_image_vae.safetensors",\n      "repo_id": "circlestone-labs/Anima",\n      "repo_file": "split_files/vae/qwen_image_vae.safetensors",\n      "revision": "f973fc41ec7545364ac9776c2440285f43ff2a30",\n      "size_bytes": 253806246,\n      "sha256": "a70580f0213e67967ee9c95f05bb400e8fb08307e017a924bf3441223e023d1f"\n    },\n    {\n      "path": "diffusion_models/waiANIMA_v10Base10.safetensors",\n      "repo_id": "kaibara/WAI-ANIMA",\n      "repo_file": "waiANIMA_v10Base10.safetensors",\n      "revision": "f328985446678f0b60e1c4e2645fd563aa69b4ff",\n      "size_bytes": 4182233976,\n      "sha256": "9d5a1e1393c2978d6a979fab38fb0dee00bc2a94e354196c9f3cf2f6f56d5fbf",\n      "source_url": "https://civitai.red/models/2544636/wai-anima",\n      "mirror_hash_matched_to_publisher": true\n    },\n    {\n      "path": "diffusion_models/AnimaYume_v15_base.safetensors",\n      "repo_id": "duongve/AnimaYume",\n      "repo_file": "split_files/diffusion_models/AnimaYume_v15_base.safetensors",\n      "revision": "e6d0f401dedb712a0a1b9620cd926ccf99c8ece3",\n      "size_bytes": 4182218328,\n      "sha256": "16354589c360b1690842c4c4d04046c35ad4de4516b15a06ac7aa350dc3265f2",\n      "source_url": "https://huggingface.co/duongve/AnimaYume"\n    },\n    {\n      "path": "diffusion_models/miaomiaoHarem_anima16.safetensors",\n      "repo_id": "icekinfe/civitai_models",\n      "repo_file": "miaomiaoHarem_anima16.safetensors",\n      "revision": "09873412000c699868b4961a678ab47a2dfbab32",\n      "size_bytes": 4182218328,\n      "sha256": "6bbb6b6785b4eb0df467658ce47f5f52f1d11fe992b1375b3c67ff02becf193b",\n      "source_url": "https://civitai.red/models/934764/miaomiao-harem?modelVersionId=3248362",\n      "mirror_hash_matched_to_publisher": true\n    }\n  ],\n  "selected_for_startup": [\n    "moss_image_sft_int8",\n    "anima_turbo_v1_1",\n    "anima_aesthetic_v1_1",\n    "anima_base_v1",\n    "wai_anima_v1_0",\n    "animayume_v1_5_base",\n    "miaomiao_harem_anima_v1_6"\n  ]\n}')
SELECTED_MODEL_IDS, MODEL_ASSETS = select_model_assets(MODEL_REGISTRY, MODEL_GROUP)
MODEL_REGISTRY["selected_for_startup"] = SELECTED_MODEL_IDS
# Prepare selected models locally and preserve verified weights on Drive.
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from importlib import metadata

COMFY_COMMIT = 'fe298e4e20f1ff26d30e8639267bbd0b57811390'
MODEL_REVISION = '24147db865fcdc207270f844777587223fed8d6e'
DRIVE_ROOT = Path('/content/drive/MyDrive/ComfyUI_Colab')
PROJECT = Path('/content') / ('ComfyUI-' + COMFY_COMMIT)
LOCAL_MODELS = Path('/content/moss_models')
MODEL_STORE = DRIVE_ROOT / 'models'
MODEL_FILES = globals().get('MODEL_ASSETS', [
 {'path':'diffusion_models/moss_image_sft_int8_convrot.safetensors','size_bytes':5413698958,'sha256':'b4d1475bc7eeffd7b809e82ea13a1fba1fb85c4e18365af3813722faa5c0fe4f'},
 {'path':'text_encoders/qwen3.5_9b_w4a8_gptq.safetensors','size_bytes':7343591088,'sha256':'4cc6a1543ad2061806da46f35f90cadb4801dcba246c44782cf42ec0feb6407c'},
 {'path':'vae/flux2-vae_small.safetensors','size_bytes':249521340,'sha256':'d8d52ba036475f5fb07c8b435e176d3d97ebfa82f0d1a1c317f9cc1e25bd013b'}])


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()


def receipt(path):
    return path.with_name(path.name + '.verified.json')


def mark_verified(path, spec):
    st = path.stat()
    receipt(path).write_text(json.dumps({'sha256':spec['sha256'], 'size_bytes':st.st_size, 'mtime_ns':st.st_mtime_ns}), encoding='utf-8')


def verified(path, spec):
    path = Path(path)
    if path.is_symlink() or not path.is_file() or path.stat().st_size != spec['size_bytes']: return False
    st = path.stat()
    expected = {'sha256':spec['sha256'], 'size_bytes':st.st_size, 'mtime_ns':st.st_mtime_ns}
    try:
        if json.loads(receipt(path).read_text(encoding='utf-8')) == expected: return True
    except (OSError, ValueError): pass
    if sha256_file(path) != spec['sha256']: return False
    mark_verified(path, spec)
    return True


def copy_verified(src, dst, spec):
    src, dst = Path(src), Path(dst)
    if src.resolve() == dst.resolve(): raise RuntimeError('模型源与目标为同一路径，已停止。')
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink(): raise RuntimeError(f'目标不能是符号链接：{dst}')
    part = dst.with_name(dst.name + '.part')
    if part.is_symlink(): raise RuntimeError(f'临时目标不能是符号链接：{part}')
    h, done, last = hashlib.sha256(), 0, time.monotonic()
    with src.open('rb') as source, part.open('wb') as target:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b''):
            target.write(chunk); h.update(chunk); done += len(chunk)
            if time.monotonic() - last > 20:
                print(f'  {dst.name}: {done / spec["size_bytes"]:.0%}', flush=True); last = time.monotonic()
        target.flush(); os.fsync(target.fileno())
    if done != spec['size_bytes'] or h.hexdigest() != spec['sha256']:
        raise RuntimeError(f'文件校验失败，原文件未替换：{dst.name}')
    os.replace(part, dst)
    mark_verified(dst, spec)


def selected_model_specs():
    # Old standalone MOSS cells only specified the canonical path and checksum.
    unique = {}
    for original in MODEL_FILES:
        spec = dict(original)
        spec.setdefault('repo_id', 'Comfy-Org/MOSS-Image')
        spec.setdefault('repo_file', spec['path'])
        spec.setdefault('revision', MODEL_REVISION)
        for field in ['path', 'repo_file']:
            relative = Path(spec[field])
            if relative.is_absolute() or '..' in relative.parts or '\\' in spec[field]:
                raise RuntimeError(f'Invalid relative model path: {spec[field]}')
        previous = unique.get(spec['path'])
        if previous is not None:
            if (previous['sha256'], previous['size_bytes']) != (spec['sha256'], spec['size_bytes']):
                raise RuntimeError(f'Conflicting assets for {spec["path"]}')
            continue
        unique[spec['path']] = spec
    return list(unique.values())


def download_model(spec, local, hf_hub_download):
    # Each source has its own resumable local_dir, separate from ComfyUI paths.
    source = spec['repo_id'] + '@' + spec['revision']
    readable = ''.join(c if c.isalnum() or c in '-_.' else '-' for c in source)[:100]
    cache = LOCAL_MODELS.parent/'model_download_cache'/(readable + '-' + hashlib.sha256(source.encode()).hexdigest()[:12])
    if cache.is_symlink(): raise RuntimeError(f'Download directory cannot be a symlink: {cache}')
    cache.mkdir(parents=True, exist_ok=True)
    candidate = cache/spec['repo_file']
    if candidate.is_symlink() or (candidate.exists() and not candidate.is_file()):
        raise RuntimeError(f'Download candidate is not a regular file: {candidate}')
    force = candidate.is_file() and not verified(candidate, spec)
    kwargs = {'repo_id': spec['repo_id'], 'filename': spec['repo_file'],
              'revision': spec['revision'], 'local_dir': str(cache)}
    downloaded = Path(hf_hub_download(**kwargs, force_download=force))
    if not verified(downloaded, spec):
        # Retry only when the returned, completed file itself is proven corrupt.
        # A corrupt canonical target must never discard a good resumable cache.
        if not force and downloaded.is_file() and not downloaded.is_symlink():
            downloaded = Path(hf_hub_download(**kwargs, force_download=True))
        if not verified(downloaded, spec):
            raise RuntimeError(f'Model download checksum failed: {spec["path"]}')
    if downloaded.resolve() != local.resolve():
        local.parent.mkdir(parents=True, exist_ok=True)
        os.replace(downloaded, local)
        mark_verified(local, spec)


def run_checked(args, **kwargs):
    subprocess.run(args, check=True, **kwargs)


def prepare_environment():
    if not Path('/content/drive/MyDrive').is_dir():
        raise RuntimeError('Drive 未挂载。请先完成第一步的 Google 授权，再重新运行全部。')
    gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True, check=True)
    print('GPU:', gpu.stdout.strip(), flush=True)
    for p in [DRIVE_ROOT, MODEL_STORE, LOCAL_MODELS, DRIVE_ROOT/'project', DRIVE_ROOT/'input', DRIVE_ROOT/'output', DRIVE_ROOT/'user']:
        if p.is_symlink(): raise RuntimeError(f'新环境目录不能是符号链接：{p}')
        p.mkdir(parents=True, exist_ok=True)
    print('[1/4] 准备 ComfyUI 固定版本', COMFY_COMMIT[:12], flush=True)
    archive = DRIVE_ROOT/'project'/f'ComfyUI-{COMFY_COMMIT}.zip'
    if not archive.is_file() or not zipfile.is_zipfile(archive):
        import urllib.request
        part = archive.with_suffix('.zip.part')
        urllib.request.urlretrieve(f'https://github.com/Comfy-Org/ComfyUI/archive/{COMFY_COMMIT}.zip', part)
        if not zipfile.is_zipfile(part): raise RuntimeError('程序下载不完整，请重新运行。')
        os.replace(part, archive)
    if PROJECT.is_symlink(): raise RuntimeError(f'本地程序目录不能是符号链接：{PROJECT}')
    marker = PROJECT/'.colab-extract-complete'
    required = [PROJECT/'main.py', PROJECT/'requirements.txt']
    try:
        complete = not marker.is_symlink() and marker.read_text(encoding='utf-8') == COMFY_COMMIT and all(p.is_file() for p in required)
    except OSError:
        complete = False
    if not complete:
        marker.unlink(missing_ok=True)
        with zipfile.ZipFile(archive) as z:
            for item in z.infolist():
                dest = (PROJECT.parent / item.filename).resolve()
                if not dest.is_relative_to(PROJECT.resolve()): raise RuntimeError('程序压缩包路径异常。')
            z.extractall(PROJECT.parent)
        if not all(p.is_file() for p in required): raise RuntimeError('ComfyUI 程序恢复失败。')
        marker_part = marker.with_name(marker.name+'.part')
        marker_part.unlink(missing_ok=True)
        marker_part.write_text(COMFY_COMMIT, encoding='utf-8')
        os.replace(marker_part, marker)
    print('[2/4] 检查依赖，保留已安装的 PyTorch/CUDA', flush=True)
    from packaging.requirements import Requirement
    missing = []
    for line in (PROJECT/'requirements.txt').read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#'): continue
        req = Requirement(line)
        if req.marker and not req.marker.evaluate(): continue
        try:
            if not req.specifier.contains(metadata.version(req.name), prereleases=True): missing.append(line)
        except metadata.PackageNotFoundError: missing.append(line)
    try: metadata.version('huggingface_hub')
    except metadata.PackageNotFoundError: missing.append('huggingface_hub')
    if missing:
        run_checked([sys.executable,'-m','pip','install','--disable-pip-version-check'] + missing)
    else: print('依赖已满足，跳过安装。', flush=True)
    specs = selected_model_specs()
    total_gb = sum(spec['size_bytes'] for spec in specs) / 1_000_000_000
    print(f'[3/4] Selected model assets: {len(specs)} files ({total_gb:.2f} GB)', flush=True)
    from huggingface_hub import hf_hub_download
    for spec in specs:
        local, saved = LOCAL_MODELS/spec['path'], MODEL_STORE/spec['path']
        print(spec['path'], flush=True)
        if not verified(local, spec):
            if verified(saved, spec):
                print('Copying verified model from Drive.', flush=True)
                copy_verified(saved, local, spec)
            else:
                print(f'Downloading from {spec["repo_id"]}.', flush=True)
                if local.is_symlink() or (local.exists() and not local.is_file()):
                    raise RuntimeError(f'Local model target is not a regular file: {local}')
                download_model(spec, local, hf_hub_download)
        else: print('Reusing verified local model.', flush=True)
        if not verified(saved, spec):
            print('Saving verified model to Drive.', flush=True)
            copy_verified(local, saved, spec)
        else: print('Drive copy is verified.', flush=True)
    manifest = {'schema_version': 1, 'files': specs}
    if 'SELECTED_MODEL_IDS' in globals(): manifest['selected_model_ids'] = SELECTED_MODEL_IDS
    sources = {(spec['repo_id'], spec['revision']) for spec in specs}
    if len(sources) == 1:
        manifest['repo_id'], manifest['revision'] = next(iter(sources))
    (DRIVE_ROOT/'model_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    if 'MODEL_REGISTRY' in globals():
        (DRIVE_ROOT/'model_registry.json').write_text(json.dumps(MODEL_REGISTRY, ensure_ascii=False, indent=2), encoding='utf-8')
    print('[4/4] 准备工作流和用户设置', flush=True)
    user = DRIVE_ROOT/'user'
    old_user = Path('/content/drive/MyDrive/ComfyUI/user')
    if old_user.is_dir() and not old_user.is_symlink() and not (user/'default').exists():
        shutil.copytree(old_user, user, dirs_exist_ok=True)
    workflows = user/'default'/'workflows'; workflows.mkdir(parents=True, exist_ok=True)
    source = None
    for package in ['comfyui-workflow-templates-json', 'comfyui-workflow-templates']:
        try: dist = metadata.distribution(package)
        except metadata.PackageNotFoundError: continue
        for file in dist.files or []:
            if Path(str(file)).name == 'image_moss_image_t2i.json':
                candidate = Path(dist.locate_file(file))
                if candidate.is_file():
                    source = candidate
                    break
        if source is not None: break
    if source is None: raise RuntimeError('未找到官方 MOSS-Image 工作流，请重新运行以检查模板依赖。')
    json.loads(source.read_text(encoding='utf-8'))
    target = workflows/'MOSS-Image_Official_INT8.json'
    try:
        json.loads(target.read_text(encoding='utf-8'))
        template_ready = True
    except (OSError, ValueError):
        template_ready = False
    if not template_ready:
        part = target.with_name(target.name+'.part')
        if part.is_symlink(): raise RuntimeError(f'工作流临时目标不能是符号链接：{part}')
        shutil.copyfile(source, part)
        os.replace(part, target)
    print('准备完成。模型从本地加载；图片、工作流和设置保存到：', DRIVE_ROOT, flush=True)
    return True


if __name__ == '__main__':
    MOSS_READY = False
    MOSS_READY = prepare_environment()


In [ ]:
# 第三步：启动 ComfyUI，并显示 Colab 内置的访问链接。
import json, os, socket, subprocess, sys, time, urllib.request
from IPython.display import display, Markdown

if not globals().get('MOSS_READY', False):
    raise RuntimeError('准备阶段未完成，请先运行前两个单元格。')
for spec in MODEL_FILES:
    if not verified(LOCAL_MODELS/spec['path'], spec):
        raise RuntimeError('本地模型校验失败，请重新运行准备阶段。')

BASE_URL = 'http://127.0.0.1:8188'
LOG_PATH = Path('/content/moss_comfyui.log')

def server_ready():
    try:
        with urllib.request.urlopen(BASE_URL+'/system_stats', timeout=2) as response:
            return 'system' in json.load(response)
    except (OSError, ValueError): return False

MOSS_PROCESS = globals().get('MOSS_PROCESS')
if MOSS_PROCESS is not None and MOSS_PROCESS.poll() is None:
    if not server_ready(): raise RuntimeError('已有进程尚未就绪，请查看 /content/moss_comfyui.log。')
    print('复用本笔记本已启动的 ComfyUI。')
else:
    with socket.socket() as sock:
        sock.settimeout(1)
        if sock.connect_ex(('127.0.0.1',8188)) == 0:
            raise RuntimeError('8188 端口已被其他服务占用，未重复启动或终止它。')
    command = [sys.executable, str(PROJECT/'main.py'), '--listen','127.0.0.1','--port','8188',
               '--models-directory',str(LOCAL_MODELS),'--user-directory',str(DRIVE_ROOT/'user'),
               '--input-directory',str(DRIVE_ROOT/'input'),'--output-directory',str(DRIVE_ROOT/'output')]
    with LOG_PATH.open('w', encoding='utf-8') as log:
        MOSS_PROCESS = subprocess.Popen(command, cwd=PROJECT, stdout=log, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 180
    while not server_ready():
        if MOSS_PROCESS.poll() is not None or time.monotonic() >= deadline:
            print(LOG_PATH.read_text(encoding='utf-8',errors='replace')[-10000:])
            raise RuntimeError('ComfyUI 启动失败或超时，请按上方日志处理后重新运行。')
        time.sleep(1)


print('ComfyUI 本地服务已就绪。下一个单元格将显示网页和 API 连接方式。')
print('ComfyUI commit:', COMFY_COMMIT)


In [ ]:
#@title 4. 云端 API（供本地提示词工具连接）
import hashlib, html, json, os, re, secrets, socket, subprocess, sys, time, urllib.request
from pathlib import Path
from IPython.display import HTML, Markdown, display

GATEWAY_SOURCE = "\"\"\"Authenticated, streaming ComfyUI gateway. Bind locally; tunnel port 8190 only.\"\"\"\nimport argparse\nimport asyncio\nimport hmac\nimport json\nimport os\nimport stat\nfrom pathlib import Path\nfrom urllib.parse import parse_qs, quote, unquote\n\nimport aiohttp\nfrom aiohttp import web\nfrom multidict import CIMultiDict\nfrom yarl import URL\n\nCOOKIE = \"moss_session\"\nHOP = {\"connection\", \"keep-alive\", \"proxy-authenticate\", \"proxy-authorization\",\n       \"te\", \"trailer\", \"transfer-encoding\", \"upgrade\"}\nSESSION = web.AppKey(\"client\", aiohttp.ClientSession)\nNO_STORE = {\"Cache-Control\": \"no-store\"}\nLOGIN = \"\"\"<!doctype html><html lang=\"zh-CN\"><meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width,initial-scale=1\">\n<title>MOSS ComfyUI Login</title><body>\n<h1>MOSS ComfyUI</h1><form method=\"post\" action=\"/_moss/login\">\n<label>API key <input type=\"password\" name=\"api_key\" maxlength=\"512\"\nrequired autocomplete=\"off\"></label><button type=\"submit\">Login</button>\n</form></body></html>\"\"\"\n\n\ndef normalize_userdata(raw_path, method):\n    \"\"\"Repair only Comfy userdata filenames decoded into paths by Colab's proxy.\"\"\"\n    path, separator, query = raw_path.partition(\"?\")\n    for prefix in (\"/userdata/\", \"/api/userdata/\"):\n        if not path.startswith(prefix):\n            continue\n        suffix = path[len(prefix):]\n        parts = suffix.split(\"/move/\", 1) if method == \"POST\" and \"/move/\" in suffix else [suffix]\n        if any(\"/\" in part for part in parts):\n            encoded = [quote(unquote(part), safe=\"\") for part in parts]\n            path = prefix + \"/move/\".join(encoded)\n        break\n    return path + separator + query\n\n\ndef same_origin(request):\n    origin = request.headers.get(\"Origin\")\n    if not origin:\n        return True\n    try:\n        parsed = URL(origin)\n        return parsed.scheme in (\"http\", \"https\") and parsed.raw_authority.lower() == request.host.lower()\n    except ValueError:\n        return False\n\n\ndef filtered_headers(headers, request=False):\n    blocked = HOP | {part.strip().lower() for part in headers.get(\"Connection\", \"\").split(\",\")}\n    if request:\n        blocked |= {\"host\", \"authorization\", \"cookie\", \"origin\", \"referer\"}\n    else:\n        blocked |= {\"set-cookie\"}\n    return CIMultiDict((key, value) for key, value in headers.items()\n                        if key.lower() not in blocked\n                        and not key.lower().startswith(\"access-control-\"))\n\n\ndef equal(value, expected):\n    try:\n        return hmac.compare_digest(value.encode(\"utf-8\"), expected.encode(\"utf-8\"))\n    except UnicodeError:\n        return False\n\n\ndef error(status, message):\n    return web.json_response({\"error\": message}, status=status, headers=NO_STORE)\n\n\nPUBLIC_MODEL_FIELDS = {\"id\", \"display_name\", \"family\", \"version\", \"assets\", \"checkpoint\",\n                       \"text_encoder\", \"vae\", \"source_url\", \"notes\"}\n\n\ndef asset_parts(value):\n    \"\"\"Accept only canonical, portable paths relative to the configured model root.\"\"\"\n    if (not isinstance(value, str) or not value\n            or any(character in value for character in \"\\\\:<>|?*\\x00\")\n            or any(ord(character) < 32 for character in value)):\n        raise ValueError(\"Invalid catalog asset path\")\n    parts = value.split(\"/\")\n    if any(part in (\"\", \".\", \"..\") or part.endswith((\".\", \" \")) for part in parts):\n        raise ValueError(\"Invalid catalog asset path\")\n    return parts\n\n\ndef asset_ready(root, name, expected_size):\n    \"\"\"Use metadata only; preparation verifies checksums before files become usable.\"\"\"\n    current = root\n    parts = asset_parts(name)\n    for index in range(len(parts) + 1):\n        if index:\n            current = current / parts[index - 1]\n        try:\n            metadata = current.lstat()\n        except (FileNotFoundError, NotADirectoryError):\n            return False\n        # Windows junctions/reparse points also redirect paths without being symlinks.\n        if (stat.S_ISLNK(metadata.st_mode)\n                or getattr(metadata, \"st_file_attributes\", 0) & getattr(stat, \"FILE_ATTRIBUTE_REPARSE_POINT\", 0)):\n            raise ValueError(\"Catalog assets must not traverse links\")\n        if index < len(parts) and not stat.S_ISDIR(metadata.st_mode):\n            return False\n    return stat.S_ISREG(metadata.st_mode) and metadata.st_size == expected_size\n\n\ndef load_model_catalog(catalog_path, model_root):\n    if not catalog_path or not model_root:\n        raise ValueError(\"Model catalog is not configured\")\n    with Path(catalog_path).open(encoding=\"utf-8\") as source:\n        catalog = json.load(source)\n    if (not isinstance(catalog, dict) or type(catalog.get(\"schema_version\")) is not int\n            or catalog[\"schema_version\"] != 1\n            or not isinstance(catalog.get(\"models\"), list)\n            or not isinstance(catalog.get(\"files\"), list)):\n        raise ValueError(\"Invalid model catalog\")\n    files = {}\n    for item in catalog[\"files\"]:\n        if not isinstance(item, dict):\n            raise ValueError(\"Invalid catalog file\")\n        name = item.get(\"path\")\n        asset_parts(name)\n        size = item.get(\"size_bytes\")\n        if name in files or type(size) is not int or size < 0:\n            raise ValueError(\"Invalid catalog file\")\n        files[name] = size\n    selected = catalog.get(\"selected_for_startup\", [])\n    if not isinstance(selected, list) or any(not isinstance(item, str) for item in selected):\n        raise ValueError(\"Invalid startup selection\")\n    models, identifiers = [], set()\n    for model in catalog[\"models\"]:\n        if not isinstance(model, dict):\n            raise ValueError(\"Invalid catalog model\")\n        identifier = model.get(\"id\")\n        if not isinstance(identifier, str) or not identifier or identifier in identifiers:\n            raise ValueError(\"Invalid catalog model ID\")\n        identifiers.add(identifier)\n        assets = model.get(\"assets\")\n        if not isinstance(assets, list) or not assets:\n            raise ValueError(\"Invalid model assets\")\n        for name in assets:\n            asset_parts(name)\n            if name not in files:\n                raise ValueError(\"Unknown model asset\")\n        for field in (\"checkpoint\", \"text_encoder\", \"vae\"):\n            if field in model and (not isinstance(model[field], str) or model[field] not in assets):\n                raise ValueError(\"Unknown model asset alias\")\n        public = {key: value for key, value in model.items() if key in PUBLIC_MODEL_FIELDS}\n        if any(not isinstance(value, str) for key, value in public.items() if key != \"assets\"):\n            raise ValueError(\"Invalid model metadata\")\n        if \"source_url\" in public:\n            source = URL(public[\"source_url\"])\n            if source.scheme != \"https\" or not source.host or source.user is not None:\n                raise ValueError(\"Invalid model source URL\")\n        defaults = model.get(\"suggested_defaults\")\n        if defaults is not None:\n            if not isinstance(defaults, dict):\n                raise ValueError(\"Invalid model defaults\")\n            public[\"suggested_defaults\"] = {\n                key: value for key, value in defaults.items()\n                if key in {\"steps\", \"cfg\", \"sampler\", \"scheduler\"}}\n            if any(not isinstance(value, (int, float, str)) for value in public[\"suggested_defaults\"].values()):\n                raise ValueError(\"Invalid model defaults\")\n        models.append(public)\n    if any(identifier not in identifiers for identifier in selected):\n        raise ValueError(\"Unknown startup model\")\n    root = Path(model_root).absolute()\n    readiness = {name: asset_ready(root, name, files[name])\n                 for name in {name for model in models for name in model[\"assets\"]}}\n    for model in models:\n        model[\"ready\"] = all(readiness[name] for name in model[\"assets\"])\n        model[\"selected_for_startup\"] = model[\"id\"] in selected\n    return {\"protocol_version\": 1, \"models\": models}\n\n\ndef create_app(upstream=\"http://127.0.0.1:8188\", api_key=None, browser_session=None,\n               catalog_path=None, model_root=None):\n    api_key = api_key if api_key is not None else os.environ.get(\"MOSS_API_KEY\", \"\")\n    browser_session = browser_session if browser_session is not None else os.environ.get(\"MOSS_BROWSER_SESSION\", \"\")\n    if not (24 <= len(api_key) <= 512 and 24 <= len(browser_session) <= 512):\n        raise ValueError(\"MOSS_API_KEY and MOSS_BROWSER_SESSION must each contain 24-512 characters\")\n    if equal(api_key, browser_session):\n        raise ValueError(\"MOSS_BROWSER_SESSION must differ from MOSS_API_KEY\")\n    target = URL(upstream)\n    if (target.scheme not in (\"http\", \"https\") or target.host not in (\"127.0.0.1\", \"::1\", \"localhost\")\n            or target.user is not None or target.path not in (\"\", \"/\") or target.query_string or target.fragment):\n        raise ValueError(\"Upstream must be an HTTP loopback origin, for example http://127.0.0.1:8188\")\n    base = str(target.origin())\n    catalog_path = catalog_path if catalog_path is not None else os.environ.get(\"MOSS_MODEL_CATALOG\", \"\")\n    model_root = model_root if model_root is not None else os.environ.get(\"MOSS_MODEL_ROOT\", \"\")\n\n    @web.middleware\n    async def authenticate(request, handler):\n        if request.path == \"/_moss/login\" and request.method in (\"GET\", \"POST\"):\n            return await handler(request)\n        header = request.headers.get(\"Authorization\", \"\")\n        if header.startswith(\"Bearer \") and equal(header[7:], api_key):\n            return await handler(request)\n        if equal(request.cookies.get(COOKIE, \"\"), browser_session):\n            if not same_origin(request):\n                return error(403, \"Cross-origin browser requests are not allowed\")\n            return await handler(request)\n        return error(401, \"API key required; use Authorization: Bearer or /_moss/login\")\n\n    async def login(request):\n        if request.method == \"GET\":\n            return web.Response(text=LOGIN, content_type=\"text/html\",\n                                headers={**NO_STORE, \"Content-Security-Policy\":\n                                         \"default-src 'none'; form-action 'self'; frame-ancestors 'none'; base-uri 'none'\"})\n        if not same_origin(request):\n            return error(403, \"Cross-origin login is not allowed\")\n        try:\n            body = await request.content.readexactly(4097)\n        except asyncio.IncompleteReadError as exc:\n            body = exc.partial\n        if len(body) > 4096:\n            return error(413, \"Login body is too large\")\n        if request.content_type != \"application/x-www-form-urlencoded\":\n            return error(415, \"Submit the login form\")\n        try:\n            submitted = parse_qs(body.decode(\"utf-8\"), max_num_fields=4).get(\"api_key\", [\"\"])[0]\n        except (ValueError, UnicodeError):\n            return error(400, \"Invalid login form\")\n        if len(submitted) > 512 or not equal(submitted, api_key):\n            return error(401, \"Invalid API key\")\n        response = web.HTTPSeeOther(\"/\", headers=NO_STORE)\n        response.set_cookie(COOKIE, browser_session, secure=True, httponly=True,\n                            samesite=\"Lax\", path=\"/\")\n        raise response\n\n    async def info(request):\n        return web.json_response({\"service\": \"moss-comfyui-gateway\", \"version\": 1,\n                                  \"auth\": \"bearer-or-browser-session\", \"upstream\": \"ComfyUI\"},\n                                 headers=NO_STORE)\n\n    async def models(request):\n        if request.method != \"GET\":\n            response = error(405, \"Model catalog supports GET only\")\n            response.headers[\"Allow\"] = \"GET\"\n            return response\n        try:\n            payload = await asyncio.to_thread(load_model_catalog, catalog_path, model_root)\n        except (OSError, ValueError, TypeError, UnicodeError):\n            return error(503, \"Model catalog is unavailable\")\n        return web.json_response(payload, headers=NO_STORE)\n\n    async def websocket(request, url, headers):\n        socket = web.WebSocketResponse(autoclose=False, max_msg_size=0)\n        if not socket.can_prepare(request).ok:\n            return error(400, \"Invalid WebSocket handshake\")\n        try:\n            remote = await request.app[SESSION].ws_connect(\n                url, headers=headers, autoclose=False, max_msg_size=0)\n        except (aiohttp.ClientError, asyncio.TimeoutError):\n            return error(502, \"ComfyUI WebSocket is unavailable\")\n        async def relay(source, destination):\n            while True:\n                message = await source.receive()\n                if message.type == aiohttp.WSMsgType.TEXT:\n                    await destination.send_str(message.data)\n                elif message.type == aiohttp.WSMsgType.BINARY:\n                    await destination.send_bytes(message.data)\n                else:\n                    code = source.close_code or 1000\n                    code = code if code not in (1005, 1006, 1015) else 1011\n                    return code, (message.extra or \"\").encode(\"utf-8\")\n\n        tasks = []\n        code, reason = 1000, b\"\"\n        try:\n            await socket.prepare(request)\n            tasks = [asyncio.create_task(relay(socket, remote)), asyncio.create_task(relay(remote, socket))]\n            done, _ = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)\n            code, reason = next(iter(done)).result()\n        except (aiohttp.ClientError, ConnectionError, RuntimeError):\n            code = 1011\n        finally:\n            for task in tasks:\n                task.cancel()\n            await asyncio.gather(*tasks, return_exceptions=True)\n            await remote.close(code=code, message=reason)\n            if socket.prepared:\n                await socket.close(code=code, message=reason)\n        return socket\n\n    async def proxy(request):\n        # encoded=True preserves model names and query values containing %2F.\n        url = URL(base + normalize_userdata(request.raw_path, request.method), encoded=True)\n        headers = filtered_headers(request.headers, request=True)\n        if request.headers.get(\"Upgrade\", \"\").lower() == \"websocket\":\n            for key in list(headers):\n                if key.lower().startswith(\"sec-websocket-\"):\n                    del headers[key]\n            return await websocket(request, url, headers)\n        response = None\n        try:\n            async with request.app[SESSION].request(\n                    request.method, url, headers=headers, allow_redirects=False,\n                    data=request.content.iter_chunked(65536) if request.can_read_body else None,\n                    skip_auto_headers={\"Content-Type\", \"Accept-Encoding\"}) as remote:\n                response = web.StreamResponse(status=remote.status, reason=remote.reason,\n                                              headers=filtered_headers(remote.headers))\n                await response.prepare(request)\n                async for chunk in remote.content.iter_chunked(65536):\n                    await response.write(chunk)\n                await response.write_eof()\n                return response\n        except (aiohttp.ClientError, asyncio.TimeoutError, ConnectionError):\n            if response is not None and response.prepared:\n                response.force_close()\n                return response\n            return error(502, \"ComfyUI is unavailable\")\n\n    async def lifecycle(app):\n        async def reject_redirect(session, context, params):\n            raise aiohttp.ClientConnectionError(\"Upstream redirects are disabled\")\n        trace = aiohttp.TraceConfig()\n        trace.on_request_redirect.append(reject_redirect)\n        timeout = aiohttp.ClientTimeout(total=None, connect=30, sock_connect=30)\n        app[SESSION] = aiohttp.ClientSession(\n            timeout=timeout, auto_decompress=False, cookie_jar=aiohttp.DummyCookieJar(),\n            trace_configs=[trace])\n        yield\n        await app[SESSION].close()\n\n    app = web.Application(middlewares=[authenticate], handler_args={\"auto_decompress\": False})\n    app.cleanup_ctx.append(lifecycle)\n    app.router.add_route(\"GET\", \"/_moss/login\", login)\n    app.router.add_route(\"POST\", \"/_moss/login\", login)\n    app.router.add_route(\"GET\", \"/_moss/info\", info)\n    app.router.add_route(\"*\", \"/_moss/models\", models)\n    app.router.add_route(\"*\", \"/{tail:.*}\", proxy)\n    return app\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--port\", type=int, default=8190)\n    parser.add_argument(\"--upstream\", default=\"http://127.0.0.1:8188\")\n    args = parser.parse_args()\n    if not 1 <= args.port <= 65535:\n        parser.error(\"--port must be between 1 and 65535\")\n    try:\n        app = create_app(args.upstream)\n    except ValueError as exc:\n        parser.error(str(exc))\n    web.run_app(app, host=\"127.0.0.1\", port=args.port, access_log=None, print=None)\n\n\nif __name__ == \"__main__\":\n    main()\n\n"
API_README = "# MOSS / ComfyUI 云端接口\n\n本文件描述云端连接协议；本地提示词工具和配套工作流由另一台电脑单独适配。\n\n## 启动和连接\n\n在 MOSS_ComfyUI_OneClick.ipynb 点击“全部运行”。最后一个单元格将创建临时 HTTPS 入口，并显示 API 地址及访问密钥。\n\n- API 根地址：最后一格显示的 https://随机名称.trycloudflare.com，不添加 /api/v1。\n- 请求头：Authorization: Bearer <本次访问密钥>。\n- 请求和返回格式：ComfyUI 原生 API。这里不是 OpenAI 的 images/generations 接口。\n- 网页登录入口：API 根地址后添加 /_moss/login。浏览器先输入访问密钥登录，使用 HttpOnly 会话 Cookie。\n- 运行时回收后，地址与密钥失效；同一运行时重复运行复用已有进程和密钥。\n- 密钥不硬编码进源码，不存入本文件；连接配置在临时文件 /content/moss_connection.json，可从 Colab 文件面板下载。\n- 笔记本关闭“保存代码输出”，避免把本次密钥长期写进笔记本。不要分享密钥或连接配置。\n- ComfyUI 和认证代理只监听运行时本机回环地址，公网隧道只转发认证代理。\n\n## 接口\n\n所有下列接口都需要认证：\n\n| 方法 | 路径 | 作用 |\n|---|---|---|\n| GET | /_moss/info | 网关协议及服务信息 |\n| GET | /_moss/models | 已接入模型、家族、建议参数、当前本地是否就绪 |\n| GET | /system_stats | ComfyUI 版本和设备状态 |\n| GET | /object_info | 节点定义、可选模型文件和输入格式 |\n| GET | /queue | 当前运行和排队任务 |\n| POST | /prompt | 提交 ComfyUI API 格式工作流，返回 prompt_id |\n| GET | /history/{prompt_id} | 查询指定任务结果 |\n| GET | /view?filename=...&subfolder=...&type=output | 下载图片，参数取自历史记录的 images 项 |\n| POST | /upload/image | multipart/form-data 上传素材 |\n| POST | /queue | 删除指定排队任务，body 为 {\"delete\":[\"prompt_id\"]} |\n| GET | /ws?clientId=... | 可选的 WebSocket 事件；原生程序可改用历史轮询 |\n\nPOST /prompt 的 body 形状为：\n\n```json\n{\"prompt\":{\"节点ID\":{\"class_type\":\"节点类型\",\"inputs\":{}}},\"client_id\":\"本机客户端随机ID\"}\n```\n\n必须提交 ComfyUI 的 API 格式 JSON（节点含 class_type 和 inputs），不能把编辑器 nodes/links JSON 直接发给 /prompt。\n\n## 接入时的最小顺序\n\n1. 用 /_moss/info 和 /system_stats 测试连接；未带密钥应得到 401。\n2. 用 /object_info 核对节点和模型文件名。\n3. 将本地构建的 MOSS API 工作流发送到 /prompt，保存返回的 prompt_id。\n4. 轮询 /history/{prompt_id}，确认 status 成功或检查执行错误。\n5. 对每个 images 条目调用 /view，保存图片。\n\nPOST /prompt 超时后不要盲目自动重试，以免重复提交。先检查队列/历史并向用户报告不确定状态。HTTP 401 表示缺少或错误密钥；连接失败通常是运行时已停止、临时地址失效或隧道尚未就绪。\n\n示例（只检查连接，不提交出图）：\n\n```python\nimport os, requests\nbase = os.environ['MOSS_API_URL'].rstrip('/')\nheaders = {'Authorization': 'Bearer ' + os.environ['MOSS_API_KEY']}\nresponse = requests.get(base + '/system_stats', headers=headers, timeout=(10, 30))\nresponse.raise_for_status()\nprint(response.json()['system'])\n```\n\n## 范围与文件\n\n权重从 /content/moss_models 加载。成品图保存在 Drive 的 ComfyUI_Colab/output；上传素材保存在 input；用户设置和已保存工作流在 user。\n\n当前官方模型文件名：\n\n- moss_image_sft_int8_convrot.safetensors\n- qwen3.5_9b_w4a8_gptq.safetensors\n- flux2-vae_small.safetensors\n\n本次仅提供协议入口与连接说明，没有修改本地提示词工具、模型配置或配套工作流。云端保留既有官方工作流。\n\n## 模型选择与目录接口\n\n准备单元提供“全部”（默认）、“仅 ANIMA”、“仅 MOSS”三个启动模型组；只影响本次要校验和恢复到本地的资产，不删除其他已缓存的模型。完整模型集11个权重共39,546,103,000字节（约39.55GB）。\n\nGET /_moss/models 返回 protocol_version=1 和 models 数组。每项含稳定 id、display_name、family、version、checkpoint、text_encoder、vae、assets、suggested_defaults、ready、selected_for_startup。资产路径相对于模型目录；填写 Comfy 节点输入时按 object_info 中的枚举文件名匹配。ready表示所有依赖在本地是普通文件且大小符合清单；完整SHA-256在准备阶段验证，不在每次目录请求时重复计算。\n\n当前稳定 ID：\n\n- moss_image_sft_int8\n- anima_turbo_v1_1\n- anima_aesthetic_v1_1\n- anima_base_v1\n- wai_anima_v1_0\n- animayume_v1_5_base\n- miaomiao_harem_anima_v1_6\n\n建议另一台电脑的出图界面先选家族（MOSS/ANIMA），再选具体模型；选项从云端目录获取，仅ready的模型可提交。MOSS与ANIMA使用不同的编码器、VAE和工作流，不能只替换主模型。suggested_defaults是起始建议；切换模型可提示采用建议值，已经手动修改的尺寸和参数应保留。所有模型共用一个API地址和密钥。\n\nWAI和MiaoMiao使用固定版本的公开镜像下载；文件大小及完整SHA-256已对照作者发布的Civitai元数据。AnimaYume和官方ANIMA来自作者仓库。具体来源、版本和校验值保存在ComfyUI_Colab/model_registry.json。不会自动替换为2.9B Beta，也没有改动本地程序或用户配套工作流。\n\n永久存储提示：Google个人账户免费云盘通常15GB，与Gmail和相册共享；这套模型约40GB不属于免费范围。取消付费存储前需迁移/减少权重，或另行选择从公开仓库下载到Colab临时磁盘的方案。\n"
CLOUDFLARED_SPEC = {'size_bytes':40227721,'sha256':'053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0'}
CLOUDFLARED_VERSION = '2026.10.3'
GATEWAY_URL = 'http://127.0.0.1:8190'


def gateway_ready(key):
    request = urllib.request.Request(GATEWAY_URL+'/_moss/info',headers={'Authorization':'Bearer '+key})
    try:
        with urllib.request.urlopen(request,timeout=3) as response:
            return json.load(response).get('service') == 'moss-comfyui-gateway'
    except (OSError,ValueError): return False


def prepare_gateway():
    global MOSS_GATEWAY_PROCESS, MOSS_API_KEY, MOSS_BROWSER_SESSION, MOSS_GATEWAY_DIGEST
    if not globals().get('MOSS_READY',False) or not server_ready():
        raise RuntimeError('ComfyUI 尚未就绪，请先完成前面的单元格。')
    api_dir = DRIVE_ROOT/'api'
    api_dir.mkdir(parents=True,exist_ok=True)
    (api_dir/'CLOUD_API.md').write_text(API_README,encoding='utf-8')
    (api_dir/'gateway.py').write_text(GATEWAY_SOURCE,encoding='utf-8')
    source_path = Path('/content/moss_gateway.py')
    digest = hashlib.sha256(GATEWAY_SOURCE.encode()).hexdigest()
    process = globals().get('MOSS_GATEWAY_PROCESS')
    if process is not None and process.poll() is None:
        if globals().get('MOSS_GATEWAY_DIGEST') != digest:
            process.terminate(); process.wait(timeout=10)
        elif gateway_ready(globals().get('MOSS_API_KEY','')):
            return
        else:
            raise RuntimeError('本笔记本的接口进程未就绪，请查看 /content/moss_gateway.log。')
    with socket.socket() as sock:
        sock.settimeout(1)
        if sock.connect_ex(('127.0.0.1',8190)) == 0:
            raise RuntimeError('8190 已被其他进程占用，未修改该进程。')
    source_path.write_text(GATEWAY_SOURCE,encoding='utf-8')
    MOSS_API_KEY = globals().get('MOSS_API_KEY') or secrets.token_urlsafe(32)
    MOSS_BROWSER_SESSION = globals().get('MOSS_BROWSER_SESSION') or secrets.token_urlsafe(32)
    environment = os.environ.copy()
    environment.update(MOSS_API_KEY=MOSS_API_KEY,MOSS_BROWSER_SESSION=MOSS_BROWSER_SESSION,
                       MOSS_MODEL_CATALOG=str(DRIVE_ROOT/'model_registry.json'),MOSS_MODEL_ROOT=str(LOCAL_MODELS))
    with Path('/content/moss_gateway.log').open('w',encoding='utf-8') as log:
        MOSS_GATEWAY_PROCESS = subprocess.Popen([sys.executable,str(source_path),'--port','8190','--upstream',BASE_URL],env=environment,stdout=log,stderr=subprocess.STDOUT)
    MOSS_GATEWAY_DIGEST = digest
    deadline = time.monotonic()+30
    while not gateway_ready(MOSS_API_KEY):
        if MOSS_GATEWAY_PROCESS.poll() is not None or time.monotonic()>deadline:
            raise RuntimeError('认证接口启动失败，请查看 /content/moss_gateway.log。')
        time.sleep(.5)


def prepare_cloudflared():
    binary = Path('/content/moss_cloudflared')
    saved = DRIVE_ROOT/'project'/('cloudflared-'+CLOUDFLARED_VERSION+'-linux-amd64')
    if not verified(binary,CLOUDFLARED_SPEC):
        if verified(saved,CLOUDFLARED_SPEC):
            copy_verified(saved,binary,CLOUDFLARED_SPEC)
        else:
            part = binary.with_name(binary.name+'.part')
            urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/download/'+CLOUDFLARED_VERSION+'/cloudflared-linux-amd64',part)
            if not verified(part,CLOUDFLARED_SPEC): raise RuntimeError('Cloudflare 官方程序校验失败。')
            os.replace(part,binary); mark_verified(binary,CLOUDFLARED_SPEC)
    if not verified(saved,CLOUDFLARED_SPEC): copy_verified(binary,saved,CLOUDFLARED_SPEC)
    binary.chmod(0o700)
    return binary


def start_cloud_api():
    global MOSS_TUNNEL_PROCESS, MOSS_API_URL
    prepare_gateway()
    process = globals().get('MOSS_TUNNEL_PROCESS')
    if process is None or process.poll() is not None:
        binary = prepare_cloudflared()
        log_path = Path('/content/moss_tunnel.log')
        with log_path.open('w',encoding='utf-8') as log:
            MOSS_TUNNEL_PROCESS = subprocess.Popen([str(binary),'--no-autoupdate','tunnel','--url',GATEWAY_URL,'--protocol','http2'],stdout=log,stderr=subprocess.STDOUT)
        MOSS_API_URL = None
    # A live tunnel may acquire its URL after a previous attempt timed out.
    log_path = Path('/content/moss_tunnel.log')
    deadline = time.monotonic()+90
    MOSS_API_URL = globals().get('MOSS_API_URL')
    while not MOSS_API_URL:
        if MOSS_TUNNEL_PROCESS.poll() is not None or time.monotonic()>deadline:
            raise RuntimeError('公网连接创建失败，请查看 /content/moss_tunnel.log 后重试。认证接口仍仅在本机监听。')
        log_text = log_path.read_text(errors='replace') if log_path.is_file() else ''
        found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log_text)
        if found: MOSS_API_URL = found.group(0)
        else: time.sleep(.5)
    # The gateway must be reachable through the tunnel before reporting success.
    deadline, last_error = time.monotonic()+60, None
    while True:
        try:
            request = urllib.request.Request(MOSS_API_URL+'/_moss/info',headers={'Authorization':'Bearer '+MOSS_API_KEY})
            with urllib.request.urlopen(request,timeout=10) as response:
                if json.load(response).get('service') != 'moss-comfyui-gateway': raise ValueError('Unexpected response')
            break
        except (OSError,ValueError) as error:
            last_error = error
            if time.monotonic()>deadline: raise RuntimeError('公网接口尚未连通，请重试最后一个单元格。') from last_error
            time.sleep(2)
    connection = {'protocol':'comfyui-http','base_url':MOSS_API_URL,'api_key':MOSS_API_KEY,'auth_header':'Authorization','auth_scheme':'Bearer','scope':'current-colab-runtime','model_catalog':'/_moss/models'}
    # This private runtime file is not copied to Drive. Download it from Colab only when needed.
    Path('/content/moss_connection.json').write_text(json.dumps(connection,ensure_ascii=False,indent=2),encoding='utf-8')
    display(Markdown('### 云端 API 已就绪\n\n**API 地址：** '+MOSS_API_URL+'\n\n网页入口：[打开 ComfyUI]('+MOSS_API_URL+'/_moss/login)，登录时输入下面的密钥。客户端请求头使用 `Authorization: Bearer <密钥>`。'))
    display(HTML('<label>本次访问密钥（点击输入框后可复制）： <input type="password" readonly autocomplete="off" style="width:420px" value="'+html.escape(MOSS_API_KEY,quote=True)+'"><button type="button" onclick="var f=this.previousElementSibling; f.type=f.type===&quot;password&quot;?&quot;text&quot;:&quot;password&quot;">显示/隐藏</button></label>'))
    print('连接配置位于 Colab 临时文件 /content/moss_connection.json。运行时回收后地址和密钥失效。')
    print('模型目录：'+MOSS_API_URL+'/_moss/models（需要相同的访问密钥）')
    print('接口文档：我的云端硬盘 / ComfyUI_Colab / api / CLOUD_API.md')


start_cloud_api()
